# Logistic Regression on 2 Datasets
Dataset 1: Heart Disease | Dataset 2: Banknote Authentication

Keep `heart.csv` and `banknote.csv` in the same folder as this notebook (or paste your own file path in Step 2).

## Step 1: Install and import

In [ ]:
!pip install pandas scikit-learn matplotlib gradio flask

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## Step 2: Set file paths
Tip: in Windows, right-click the file > Copy as path, then paste it like `r"C:\Users\Name\Downloads\heart.csv"`

In [ ]:
path1 = "heart.csv"
path2 = "banknote.csv"

## Dataset 1: Heart Disease

In [ ]:
df1 = pd.read_csv(path1)
df1.head()

In [ ]:
df1["HeartDisease"].value_counts().plot(kind="bar", color=["green", "red"])
plt.title("Heart Disease: Class Count (0 = No, 1 = Yes)")
plt.xlabel("HeartDisease")
plt.ylabel("Number of patients")
plt.show()

In [ ]:
X1 = df1.drop("HeartDisease", axis=1)
y1 = df1["HeartDisease"]
features1 = list(X1.columns)

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

model1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model1.fit(X1_train, y1_train)

pred1 = model1.predict(X1_test)
print("Accuracy:", accuracy_score(y1_test, pred1))
print(classification_report(y1_test, pred1))

In [ ]:
ConfusionMatrixDisplay.from_estimator(model1, X1_test, y1_test, cmap="Blues")
plt.title("Heart Disease: Confusion Matrix")
plt.show()

In [ ]:
plt.barh(features1, model1[-1].coef_[0])
plt.title("Heart Disease: Feature Importance (Coefficients)")
plt.show()

## Dataset 2: Banknote Authentication

In [ ]:
df2 = pd.read_csv(path2)
df2.head()

In [ ]:
df2["Class"].value_counts().plot(kind="bar", color=["green", "red"])
plt.title("Banknote: Class Count (0 = Genuine, 1 = Forged)")
plt.xlabel("Class")
plt.ylabel("Number of notes")
plt.show()

In [ ]:
X2 = df2.drop("Class", axis=1)
y2 = df2["Class"]
features2 = list(X2.columns)

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model2.fit(X2_train, y2_train)

pred2 = model2.predict(X2_test)
print("Accuracy:", accuracy_score(y2_test, pred2))
print(classification_report(y2_test, pred2))

In [ ]:
ConfusionMatrixDisplay.from_estimator(model2, X2_test, y2_test, cmap="Blues")
plt.title("Banknote: Confusion Matrix")
plt.show()

In [ ]:
plt.barh(features2, model2[-1].coef_[0])
plt.title("Banknote: Feature Importance (Coefficients)")
plt.show()

## Gradio UI (runs at http://127.0.0.1:7860)

In [ ]:
import gradio as gr

def predict_heart(age, sex, cp, bp, chol, thalach, exang, oldpeak):
    data = pd.DataFrame([[age, sex, cp, bp, chol, thalach, exang, oldpeak]], columns=features1)
    prob = model1.predict_proba(data)[0][1]
    return ("Heart Disease" if prob >= 0.5 else "No Heart Disease") + f" (probability of disease: {prob:.2f})"

def predict_banknote(variance, skewness, curtosis, entropy):
    data = pd.DataFrame([[variance, skewness, curtosis, entropy]], columns=features2)
    prob = model2.predict_proba(data)[0][1]
    return ("Forged" if prob >= 0.5 else "Genuine") + f" (probability of forged: {prob:.2f})"

ui1 = gr.Interface(
    fn=predict_heart,
    inputs=[gr.Number(label=f, value=v) for f, v in zip(features1, [55, 1, 1, 130, 240, 150, 0, 1.0])],
    outputs="text",
    title="Heart Disease Prediction")

ui2 = gr.Interface(
    fn=predict_banknote,
    inputs=[gr.Number(label=f, value=v) for f, v in zip(features2, [3.6, 8.7, -2.8, -0.4])],
    outputs="text",
    title="Banknote Authentication")

demo = gr.TabbedInterface([ui1, ui2], ["Heart Disease", "Banknote"])
demo.launch()

## Flask UI (runs at http://127.0.0.1:5000)

In [ ]:
from flask import Flask, request, render_template_string
from threading import Thread

app = Flask(__name__)

page = """
<h2>{{ title }}</h2>
<form method="POST">
  {% for f in features %}
    {{ f }}: <input type="number" step="any" name="{{ f }}" required><br><br>
  {% endfor %}
  <button type="submit">Predict</button>
</form>
{% if result %}<h3>Result: {{ result }}</h3>{% endif %}
<br><a href="/">Home</a>
"""

@app.route("/")
def home():
    return '<h1>Logistic Regression App</h1><a href="/heart">Heart Disease</a><br><br><a href="/banknote">Banknote</a>'

@app.route("/heart", methods=["GET", "POST"])
def heart():
    result = None
    if request.method == "POST":
        data = pd.DataFrame([[float(request.form[f]) for f in features1]], columns=features1)
        result = "Heart Disease" if model1.predict(data)[0] == 1 else "No Heart Disease"
    return render_template_string(page, title="Heart Disease Prediction", features=features1, result=result)

@app.route("/banknote", methods=["GET", "POST"])
def banknote():
    result = None
    if request.method == "POST":
        data = pd.DataFrame([[float(request.form[f]) for f in features2]], columns=features2)
        result = "Forged" if model2.predict(data)[0] == 1 else "Genuine"
    return render_template_string(page, title="Banknote Authentication", features=features2, result=result)

Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()